In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

DATA_PATH = "/content/drive/MyDrive/datasets/final_dataset.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ Загружено примеров: {len(data)}")
print("Пример:", data[0])

✅ Загружено примеров: 10529
Пример: {'messages': [{'role': 'user', 'content': 'Напиши текст в научном стиле. Тема: Подмножество'}, {'role': 'assistant', 'content': 'В математике говорят, что множество есть подмно́жество множества , если все элементы первого множества являются и элементами второго множества. Определение Множество называется подмножеством множества , если все элементы, принадлежащие , также принадлежат . Формальное определение: Существует две системы символических обозначений для подмножеств: Обе системы обозначений предусмотрены стандартом ISO 31-11, но используют символ в разных смыслах, что может привести к путанице. В данной статье мы будем использовать последнюю систему обозначений. Множество называется множества , если является подмножеством множества . То, что является надмножеством множества , записывают , то есть Множество всех подмножеств множества обозначается . Множества и называются равными , только когда они состоят из одних и тех же элементов, то есть и .'

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print("✅ Модель загружена")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

✅ Модель загружена


In [ ]:
from datasets import Dataset

def format_chat(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

formatted_texts = [format_chat(x) for x in data]

dataset = Dataset.from_dict({"text": formatted_texts})

print("✅ Пример после форматирования:\n")
print(formatted_texts[0][:500])

✅ Пример после форматирования:

<|user|>
Напиши текст в научном стиле. Тема: Подмножество<|end|>
<|assistant|>
В математике говорят, что множество есть подмно́жество множества , если все элементы первого множества являются и элементами второго множества. Определение Множество называется подмножеством множества , если все элементы, принадлежащие , также принадлежат . Формальное определение: Существует две системы символических обозначений для подмножеств: Обе системы обозначений предусмотрены стандартом ISO 31-11, но используют


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
    )

dataset = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=2,
)

Map (num_proc=2):   0%|          | 0/10529 [00:00<?, ? examples/s]

In [ ]:
dataset_split = dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print(f"Train: {len(train_dataset)}")
print(f"Eval: {len(eval_dataset)}")

Train: 10002
Eval: 527


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # ⚠️ оптимально для Colab (не 64!)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from transformers import TrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/phi3_finetune"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # = batch 8

    num_train_epochs=2,  # ⚠️ сначала 1-2, не 3
    learning_rate=2e-4,

    logging_steps=10,  # 🔥 частые логи
    save_strategy="steps",
    save_steps=100,  # 🔥 сохраняем часто!
    save_total_limit=3,

    eval_strategy="steps",
    eval_steps=50,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",

    report_to="none",
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,002 | Num Epochs = 2 | Total steps = 2,502
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,2.315063,2.307002
100,2.193136,2.237328
150,2.268862,2.200340
200,2.155212,2.170120
250,2.214211,2.148647
300,2.187903,2.127455
350,2.092102,2.113540
400,2.150500,2.094942
450,2.117137,2.086037
500,2.007817,2.069813


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=2502, training_loss=1.9243253757246583, metrics={'train_runtime': 12808.7268, 'train_samples_per_second': 1.562, 'train_steps_per_second': 0.195, 'total_flos': 1.4803954274451456e+17, 'train_loss': 1.9243253757246583, 'epoch': 2.0})

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('/content/drive/MyDrive/phi3_finetune/tokenizer_config.json',
 '/content/drive/MyDrive/phi3_finetune/chat_template.jinja',
 '/content/drive/MyDrive/phi3_finetune/tokenizer.json')

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/phi3_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Downloaded tokenizer.model



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:03<00:00, 91.79s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/phi3_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/phi3_gguf_gguf/phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/phi3_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/phi3_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/phi3_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/phi3_gguf_gguf/Modelfile


{'save_directory': '/content/drive/MyDrive/phi3_gguf',
 'gguf_directory': '/content/drive/MyDrive/phi3_gguf_gguf',
 'gguf_files': ['/content/drive/MyDrive/phi3_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/drive/MyDrive/phi3_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}